In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

In [ ]:
try:
    url = "https://raw.githubusercontent.com/UBC-MDS/maternal_health_risk_predictor/main/data/maternal_health_risk.csv"
    df = pd.read_csv(url)
    print("Dataset loaded successfully from online source!")
except:
    print("Online source not available. Loading from local file...")
    print("Please ensure 'Maternal Health Risk Data Set.csv' is in the same folder")
    df = pd.read_csv('Maternal Health Risk Data Set.csv')
    print("Dataset loaded successfully from local file!")

print("Total number of patients:", len(df))
print("Number of features:", len(df.columns) - 1)
print()

print("Sample data (first 5 rows):")
print(df.head())
print()

Online source not available. Loading from local file...
Please ensure 'Maternal Health Risk Data Set.csv' is in the same folder
Dataset loaded successfully from local file!
Total number of patients: 1014
Number of features: 6

Sample data (first 5 rows):
   Age  SystolicBP  DiastolicBP    BS  BodyTemp  HeartRate  RiskLevel
0   25         130           80  15.0      98.0         86  high risk
1   35         140           90  13.0      98.0         70  high risk
2   29          90           70   8.0     100.0         80  high risk
3   30         140           85   7.0      98.0         70  high risk
4   35         120           60   6.1      98.0         76   low risk



In [ ]:
print("Column names in dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")
print()

print("Target variable (what we want to predict): RiskLevel")
print("Risk level distribution:")
print(df['RiskLevel'].value_counts())
print()

Column names in dataset:
  1. Age
  2. SystolicBP
  3. DiastolicBP
  4. BS
  5. BodyTemp
  6. HeartRate
  7. RiskLevel

Target variable (what we want to predict): RiskLevel
Risk level distribution:
RiskLevel
low risk     406
mid risk     336
high risk    272
Name: count, dtype: int64



In [ ]:
X = df.drop('RiskLevel', axis=1)
y = df['RiskLevel']

print("Features (X) - Patient health parameters:")
print(list(X.columns))
print()

print("Converting risk levels to binary classification:")
print("  Original: 'low risk', 'mid risk', 'high risk'")
print("  New:      'Low Risk' (0), 'High Risk' (1)")
print()

y_binary = y.apply(lambda x: 1 if 'high' in x.lower() else 0)

print("Distribution after conversion:")
print(f"  Low Risk (0):  {(y_binary == 0).sum()} patients")
print(f"  High Risk (1): {(y_binary == 1).sum()} patients")
print()

Features (X) - Patient health parameters:
['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']

Converting risk levels to binary classification:
  Original: 'low risk', 'mid risk', 'high risk'
  New:      'Low Risk' (0), 'High Risk' (1)

Distribution after conversion:
  Low Risk (0):  742 patients
  High Risk (1): 272 patients



In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

print(f"Training set: {len(X_train)} patients (80%)")
print(f"Testing set:  {len(X_test)} patients (20%)")
print()

Training set: 811 patients (80%)
Testing set:  203 patients (20%)



In [ ]:
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)
print("Missing values handled (if any)")

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Features scaled to standard range")
print()

Missing values handled (if any)
Features scaled to standard range



In [26]:
print("Original training data:")
print(f"  Low Risk:  {(y_train == 0).sum()} patients")
print(f"  High Risk: {(y_train == 1).sum()} patients")
print()
print("Using SMOTE to balance classes during training")
print("SMOTE creates synthetic examples of minority class")
print()

Original training data:
  Low Risk:  593 patients
  High Risk: 218 patients

Using SMOTE to balance classes during training
SMOTE creates synthetic examples of minority class



In [ ]:
print("We will train 4 different models and compare them:")
print("  1. Logistic Regression - Simple linear model")
print("  2. Support Vector Machine - Finds optimal boundary")
print("  3. Random Forest - Uses multiple decision trees")
print("  4. XGBoost - Advanced boosting algorithm")
print()

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Support Vector Machine': SVC(kernel='rbf', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

results = []
trained_models = {}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for model_name, model in models.items():
    print(f"Training: {model_name}...")

    pipeline = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('classifier', model)
    ])

    accuracy_scores = cross_val_score(pipeline, X_train, y_train,
                                       cv=cv, scoring='accuracy')
    f1_scores = cross_val_score(pipeline, X_train, y_train,
                                 cv=cv, scoring='f1')

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

    test_accuracy = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_pred_proba)

    results.append({
        'Model': model_name,
        'Accuracy': test_accuracy,
        'F1-Score': test_f1,
        'ROC-AUC': test_auc
    })

    trained_models[model_name] = pipeline

    print(f"  Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
    print(f"  F1-Score: {test_f1:.4f}")
    print()

We will train 4 different models and compare them:
  1. Logistic Regression - Simple linear model
  2. Support Vector Machine - Finds optimal boundary
  3. Random Forest - Uses multiple decision trees
  4. XGBoost - Advanced boosting algorithm

Training: Logistic Regression...
  Accuracy: 0.8719 (87.19%)
  F1-Score: 0.7797

Training: Support Vector Machine...
  Accuracy: 0.8867 (88.67%)
  F1-Score: 0.8034

Training: Random Forest...
  Accuracy: 0.9507 (95.07%)
  F1-Score: 0.9091

Training: XGBoost...
  Accuracy: 0.9557 (95.57%)
  F1-Score: 0.9174



In [27]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Accuracy', ascending=False)

print(results_df.to_string(index=False))
print()

best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
best_accuracy = results_df.iloc[0]['Accuracy']

print(f"BEST MODEL: {best_model_name}")
print(f"Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
print()

                 Model  Accuracy  F1-Score  ROC-AUC
               XGBoost  0.955665  0.917431 0.967686
         Random Forest  0.950739  0.909091 0.971663
Support Vector Machine  0.886700  0.803419 0.911882
   Logistic Regression  0.871921  0.779661 0.929530

BEST MODEL: XGBoost
Accuracy: 0.9557 (95.57%)



In [ ]:
y_pred = best_model.predict(X_test)

print("Classification Report:")
print("(Shows precision, recall, f1-score for each class)")
print()
print(classification_report(y_test, y_pred,
                           target_names=['Low Risk', 'High Risk']))

print("Confusion Matrix:")
print("(Shows correct and incorrect predictions)")
print()
cm = confusion_matrix(y_test, y_pred)
print("                Predicted Low   Predicted High")
print(f"Actual Low          {cm[0,0]:4d}            {cm[0,1]:4d}")
print(f"Actual High         {cm[1,0]:4d}            {cm[1,1]:4d}")
print()

tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print("Clinical Metrics:")
print(f"  Sensitivity: {sensitivity:.4f} ({sensitivity*100:.1f}%)")
print(f"    - Out of all high-risk patients, we correctly identified {sensitivity*100:.1f}%")
print()
print(f"  Specificity: {specificity:.4f} ({specificity*100:.1f}%)")
print(f"    - Out of all low-risk patients, we correctly identified {specificity*100:.1f}%")
print()

Classification Report:
(Shows precision, recall, f1-score for each class)

              precision    recall  f1-score   support

    Low Risk       0.97      0.97      0.97       149
   High Risk       0.91      0.93      0.92        54

    accuracy                           0.96       203
   macro avg       0.94      0.95      0.94       203
weighted avg       0.96      0.96      0.96       203

Confusion Matrix:
(Shows correct and incorrect predictions)

                Predicted Low   Predicted High
Actual Low           144               5
Actual High            4              50

Clinical Metrics:
  Sensitivity: 0.9259 (92.6%)
    - Out of all high-risk patients, we correctly identified 92.6%

  Specificity: 0.9664 (96.6%)
    - Out of all low-risk patients, we correctly identified 96.6%



In [ ]:
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print("Saved: best_model.pkl")

with open('imputer.pkl', 'wb') as f:
    pickle.dump(imputer, f)
print("Saved: imputer.pkl")

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Saved: scaler.pkl")
print()

Saved: best_model.pkl
Saved: imputer.pkl
Saved: scaler.pkl



In [ ]:
print()

def predict_risk(age, systolic_bp, diastolic_bp, blood_sugar, body_temp, heart_rate):
    """
    Predicts maternal health risk for a patient

    Input:
        age: Patient's age in years
        systolic_bp: Systolic blood pressure (mmHg)
        diastolic_bp: Diastolic blood pressure (mmHg)
        blood_sugar: Blood glucose level (mmol/L)
        body_temp: Body temperature (Fahrenheit)
        heart_rate: Heart rate (beats per minute)

    Output:
        Risk prediction: "Low Risk" or "High Risk"
        Probability: Chance of being high risk (0 to 1)
    """

    patient_data = np.array([[age, systolic_bp, diastolic_bp,
                             blood_sugar, body_temp, heart_rate]])

    patient_data = imputer.transform(patient_data)
    patient_data = scaler.transform(patient_data)

    prediction = best_model.predict(patient_data)[0]
    probability = best_model.predict_proba(patient_data)[0, 1]

    risk = "Low Risk" if prediction == 0 else "High Risk"

    return risk, probability


print("Example Predictions:")
print()

print("Case 1: Normal Healthy Patient")
print("  Age: 28, BP: 115/75, Blood Sugar: 6.5")
print("  Body Temp: 98.2°F, Heart Rate: 75 bpm")
risk, prob = predict_risk(28, 115, 75, 6.5, 98.2, 75)
print(f"  Prediction: {risk}")
print(f"  High Risk Probability: {prob:.2%}")
print()

print("Case 2: Patient with High Blood Pressure")
print("  Age: 35, BP: 140/90, Blood Sugar: 7.8")
print("  Body Temp: 98.5°F, Heart Rate: 88 bpm")
risk, prob = predict_risk(35, 140, 90, 7.8, 98.5, 88)
print(f"  Prediction: {risk}")
print(f"  High Risk Probability: {prob:.2%}")
print()

print("Case 3: High Risk Patient")
print("  Age: 42, BP: 160/105, Blood Sugar: 10.5")
print("  Body Temp: 99.0°F, Heart Rate: 95 bpm")
risk, prob = predict_risk(42, 160, 105, 10.5, 99.0, 95)
print(f"  Prediction: {risk}")
print(f"  High Risk Probability: {prob:.2%}")
print()


Example Predictions:

Case 1: Normal Healthy Patient
  Age: 28, BP: 115/75, Blood Sugar: 6.5
  Body Temp: 98.2°F, Heart Rate: 75 bpm
  Prediction: Low Risk
  High Risk Probability: 0.14%

Case 2: Patient with High Blood Pressure
  Age: 35, BP: 140/90, Blood Sugar: 7.8
  Body Temp: 98.5°F, Heart Rate: 88 bpm
  Prediction: High Risk
  High Risk Probability: 99.84%

Case 3: High Risk Patient
  Age: 42, BP: 160/105, Blood Sugar: 10.5
  Body Temp: 99.0°F, Heart Rate: 95 bpm
  Prediction: High Risk
  High Risk Probability: 99.88%

